# Model name experimentation
This notebook contains the experiments regarding the detection of datasets from the papers

In [3]:
import sys
sys.path.append('/home/jovyan/BenchmarkingML4KGE_extraction')

from utils.XMLParser import XMLParser
from utils.grobid_service import GrobidService
from utils import experimentation_utils
import json
from tqdm.auto import tqdm
from bert_score import score
import torch
import time
from gliner import GLiNER
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

with open('../data/dataset_con_rutas_xml.json', 'r', encoding='utf-8') as f:
    kge_dataset=json.load(f)

service=GrobidService()


/home/jovyan/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/jovyan/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/jovyan/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
2026-02-26 13:55:22.209131962 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file

# GliNER

In [2]:
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
device= "cuda" if torch.cuda.is_available() else "cpu"
labels = ["task"]

/home/jovyan/.local/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt= []
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")

    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_full_text()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    init_time=time.time()

    entities=gliner_model.predict_entities(text,labels,threshold=0.4)
    predictions=list(set(ent['text'] for ent in entities))

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)


Processing:   0%|          | 0/89 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 6552 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 7266 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 11953 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.3519 seg
🎯 BERTScore F1 promedio: 0.6381


## Only abstract


In [4]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt= []
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")

    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_abstract()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    init_time=time.time()

    entities=gliner_model.predict_entities(text,labels,threshold=0.4)
    predictions=list(set(ent['text'] for ent in entities))

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 572 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.1519 seg
🎯 BERTScore F1 promedio: 0.6213


## Sectioned

In [5]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt= []
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")

    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    sections=parser.get_sections(target_sections=['Methodology','Model','Embedding'])
    text='\n'.join(sections.values())

    if not text or not gt:
        continue
        print('An error occurred while processing')

    init_time=time.time()

    entities=gliner_model.predict_entities(text,labels,threshold=0.4)
    predictions=list(set(ent['text'] for ent in entities))

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1653 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 650 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 907 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 3984 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1505 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 781 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.2459 seg
🎯 BERTScore F1 promedio: 0.2485


# Model 2: Qwen3:1.7B

In [4]:
import os
from ollama import Client
from utils import llm_preprocessing
from transformers import AutoModelForCausalLM, AutoTokenizer
import accelerate

model_name = "Qwen/Qwen3-1.7B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
max_context_tokens = 32768 - 2048



Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## Full paper

In [5]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

question="What is the name of the model presented in this paper?"
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=[]
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_full_text()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.8676 seg
🎯 BERTScore F1 promedio: 0.7623


## Abstract

In [6]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

question="What is the name of the model presented in this paper?"
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=[]
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_abstract()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.1383 seg
🎯 BERTScore F1 promedio: 0.5724


## Sections

In [7]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

question="What is the name of the model presented in this paper?"
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=[]
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    sections=parser.get_sections(target_sections=['Methodology','Model','Embedding'])
    text='\n'.join(sections.values())

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.0883 seg
🎯 BERTScore F1 promedio: 0.1184


# Model 3: Llama 3

In [8]:
import os
from pathlib import Path
import ollama
from ollama import Client, ResponseError
resultados = []
tiempos = []
scores_f1 = []

model_name="llama3"
question="What is the name of the model presented in this paper?"
base_path = Path(os.getcwd()).parent

client=Client(timeout=600.0)

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=[]
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_full_text()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error")
    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 1.8886 seg
🎯 BERTScore F1 promedio: 0.7436


## Abstract

In [10]:
import os
from pathlib import Path
from ollama import Client, ResponseError

client=Client(timeout=600.0)

resultados = []
tiempos = []
scores_f1 = []

model_name="llama3"
question="What is the name of the model presented in this paper?"
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=[]
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_abstract()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error")

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.3214 seg
🎯 BERTScore F1 promedio: 0.7516


## Sections

In [12]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

model_name="llama3"
question="What is the name of the model presented in this paper?"
base_path = Path(os.getcwd()).parent

client=Client(timeout=600.0)


for paper in tqdm(kge_dataset, desc="Processing"):
    gt=[]
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        gt.append(method_name)
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    sections=parser.get_sections(target_sections=['Methodology','Model','Embedding'])
    text='\n'.join(sections.values())

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error")

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.4858 seg
🎯 BERTScore F1 promedio: 0.7397
